In [1]:
from pybtex.database.input import bibtex
from time import strptime
import string
import html
import os
import re


In [2]:

# --- CONFIGURATION ---
# Your name for bolding in author lists
MY_LAST_NAME = "Sankagiri"

# Input/Output settings
PUB_LIST = {
    "all": {
        "file": "suryanarayana_sankagiri_allpubs_abstracts.bib",
        "collection": {"name": "publications", "permalink": "/publication/"}
    }
}

# Categories for sorting
ALLOWED_CATEGORIES = {"preprints", "journals", "conferences", "workshops", "books"}

# Venue Acronym Mappings
VENUE_MAPPING = {
    "ALT": "International Conference on Algorithmic Learning Theory (ALT)",
    "ICML": "International Conference on Machine Learning (ICML)",
    "UAI": "Conference on Uncertainty in Artificial Intelligence (UAI)",
    "SCC": "IEEE International Conference on Services Computing (SCC)",
    "NCC": "National Conference on Communications (NCC)",
    "ISIT": "IEEE International Symposium on Information Theory (ISIT)",
    "ISMIR": "International Society for Music Information Retrieval Conference (ISMIR)",
    "Blockchain": "IEEE International Conference on Blockchain",
    "ConsensusDay@CCS": "ConsensusDay Workshop at ACM CCS",
}

print("Configuration Loaded.")

Configuration Loaded.


In [3]:
# --- HELPER FUNCTIONS ---

def html_escape(text):
    """Produces HTML entities within text."""
    html_escape_table = {
        "&": "&amp;",
        '"': "&quot;",
        "'": "&apos;"
    }
    return "".join(html_escape_table.get(c,c) for c in text)

In [4]:
def get_category(fields):
    """Determines category (journal, conference, etc.) from keywords."""
    kw = fields.get("keywords", "").strip().lower()
    parts = [p.strip() for p in kw.replace(";", ",").split(",") if p.strip()]
    for p in parts:
        if p in ALLOWED_CATEGORIES:
            return p
    return "preprints" # Default

In [5]:
def format_authors(authors_list):
    """Formats author list, bolding the configured name."""
    formatted_names = []
    for author in authors_list:
        # Get parts and filter empty strings
        first = " ".join(author.first_names)
        middle = " ".join(author.middle_names)
        last = " ".join(author.last_names)
        
        name = " ".join(filter(None, [first, middle, last]))
        name = name.replace("{", "").replace("}", "").replace("\\", "")
        
        if MY_LAST_NAME in last:
            name = f"<b>{name}</b>"
        
        formatted_names.append(name)
    return ", ".join(formatted_names)

In [6]:
def clean_venue_name(fields):
    """Extracts and cleans the venue name using the mapping."""
    raw = fields.get("booktitle", fields.get("journal", ""))
    clean = raw.replace("{", "").replace("}", "").replace("\\", "")
    return VENUE_MAPPING.get(clean, clean)

In [7]:
def parse_date(fields):
    """Parses year/month/day into a YYYY-MM-DD string."""
    year = fields.get("year", "1900")
    month = fields.get("month", "01")
    day = fields.get("day", "01")
    
    # Handle month abbreviations (e.g., 'jan') or numbers
    if len(month) < 3:
        month = month.zfill(2) # '1' -> '01'
    elif month.lower() not in ["01","02","03","04","05","06","07","08","09","10","11","12"]:
        try:
            tmnth = strptime(month[:3],'%b').tm_mon   
            month = "{:02d}".format(tmnth) 
        except:
            month = "01"
            
    return f"{year}-{month}-{day}"

print("Helper Functions Loaded.")

Helper Functions Loaded.


In [8]:
# --- MAIN EXECUTION ---

for pubsource in PUB_LIST:
    parser = bibtex.Parser()
    bibdata = parser.parse_file(PUB_LIST[pubsource]["file"])

    for bib_id in bibdata.entries:
        b = bibdata.entries[bib_id].fields
        
        try:
            # 1. Prepare Data using Helpers
            pub_date = parse_date(b)
            venue = clean_venue_name(b)
            category = get_category(b)
            
            # Author formatting
            author_str = ""
            if "author" in bibdata.entries[bib_id].persons:
                author_str = format_authors(bibdata.entries[bib_id].persons["author"])

            # 2. Prepare Filename & Title
            raw_title = b["title"].replace("{", "").replace("}", "").replace("\\", "")
            clean_title_slug = raw_title.replace(" ", "-")
            
            url_slug = re.sub("\\[.*\\]|[^a-zA-Z0-9_-]", "", clean_title_slug).replace("--", "-")
            filename = f"{pub_date}-{url_slug}.md"
            
            # 3. Build Markdown Content (YAML Header)
            md = "---\n"
            # Now we use the pre-cleaned 'raw_title' variable here
            md += f"title: \"{html_escape(raw_title)}\"\n"
            
            md += f"collection: {PUB_LIST[pubsource]['collection']['name']}\n"
            md += f"permalink: {PUB_LIST[pubsource]['collection']['permalink']}{pub_date}-{url_slug}\n"
            md += f"authors: '{html_escape(author_str)}'\n"
            md += f"date: {pub_date}\n"
            md += f"venue: '{html_escape(venue)}'\n"
            md += f"category: {category}\n"
            md += "excerpt: ''\n"

            # --- DUAL LINK LOGIC ---
            # 'paperurl' = ArXiv PDF (Standard Button)
            if "pdf" in b and len(str(b["pdf"])) > 5:
                md += f"paperurl: '{b['pdf']}'\n"
            
            # 'officialurl' = Publisher Link (New Button)
            if "url" in b and len(str(b["url"])) > 5:
                md += f"officialurl: '{b['url']}'\n"
                
            md += "---\n"

            # 4. Markdown Body (Abstract + Links)
            if "abstract" in b:
                md += f"\n{b['abstract'].strip()}\n"

            # Add clickable text links at bottom of abstract (optional redundancy)
            if "pdf" in b:
                md += f"\n[Download PDF]({b['pdf']}){{: .btn .btn--info}} "
            if "url" in b:
                md += f"\n[Publisher Page]({b['url']}){{:target=\"_blank\"}}\n" 

            # 5. Write File
            with open(f"../_publications/{filename}", 'w', encoding="utf-8") as f:
                f.write(md)
            
            print(f"SUCCESS: {bib_id}")

        except KeyError as e:
            print(f"WARNING: Missing Field {e} in {bib_id}")
            continue

SUCCESS: DBLP:conf/alt/SankagiriEFG26
SUCCESS: DBLP:conf/alt/VillemaudSG26
SUCCESS: DBLP:conf/icml/SankagiriEG25
SUCCESS: DBLP:conf/uai/CorreaSFG25
SUCCESS: DBLP:journals/ton/SankagiriH25
SUCCESS: DBLP:journals/sigmetrics/SankagiriH24
SUCCESS: DBLP:conf/IEEEscc/SampathNDDS22
SUCCESS: DBLP:conf/consensusday/AmeenSH22
SUCCESS: DBLP:conf/fc/SankagiriWKV21
SUCCESS: DBLP:conf/fc/BaileyS21
SUCCESS: DBLP:journals/ss/SankagiriGH23
SUCCESS: DBLP:conf/blockchain2/SampathDKNDS20
SUCCESS: DBLP:journals/tit/HajekS19
SUCCESS: DBLP:conf/isit/HajekS18
SUCCESS: DBLP:conf/ismir/PSGR16
SUCCESS: DBLP:conf/ncc/PSR16
